In [1]:
# %pip install pandas-3.1.1

In [2]:
# Import libraries

import json
import pandas as pd
import ollama
import autogen

from autogen import ConversableAgent, initiate_chats

In [3]:
# Configure Ollama (the model returns its response all at once or gradually in chunks (stream).)

llm_config = {"config_list": [{"model": "qwen3:1.7b", "api_type": "ollama", 
                               "client_host": "http://localhost:11434", "stream": False}] , "temperature": 0.03}

In [4]:
# Create Personal Information Agent

onboarding_personal_information_agent = ConversableAgent(name = "onboarding_personal_information_agent",
                                                         system_message = """
                                                         You are a customer onboarding agent.
                                                         Your responsibility is to collect:
                                                         1. Customer name
                                                         2. Customer location
                                                         Ask the customer for these details.
                                                         Once the customer provides both details,
                                                         respond in this exact format:
                                                         CUSTOMER_DATA:{"name": "customer name", "location": "customer location"}
                                                         Do not invent customer information.
                                                         End your response with TERMINATE.
                                                         """, llm_config = llm_config, code_execution_config = False, 
                                                         human_input_mode = "NEVER")

In [5]:
# Create Topic Preference Agent

onboarding_topic_preference_agent = ConversableAgent(name = "onboarding_topic_preference_agent", 
                                                     system_message = """
                                                     You are a customer topic preference agent.
                                                     Your responsibility is to collect the customer's
                                                     preferred reading topics
                                                     Examples:
                                                     Technology, Sports, Thriller, Business, AI.
                                                     Once the customer provides their preference,
                                                     respond in this format:
                                                     TOPIC_PREFERENCE: customer preference
                                                     Preserve the customer's actual preference.
                                                     Do not invent a different preference.
                                                     End your response with TERMINATE.
                                                     """, llm_config = llm_config, code_execution_config = False, 
                                                     human_input_mode = "NEVER")

In [6]:
# Create Customer Engagement Agent

customer_engagement_agent = ConversableAgent(name="customer_engagement_agent",
                                             system_message="""
                                             You are a customer engagement agent.
                                             Your responsibility is to provide personalized
                                             content based on customer information and preferences.
                                             You will receive:
                                             1. Customer name
                                             2. Customer location
                                             3. Customer topic preference
                                             Follow these instructions:
                                             - Address the customer by their name.
                                             - Understand their actual topic preference.
                                             - Generate relevant and interesting content.
                                             - If the preference is Thriller, provide a short thriller story.
                                             - If the preference is Technology, provide interesting technology information.
                                             - Do not generate unrelated content.
                                             - Never invent customer information.
                                             End your response with TERMINATE.
                                             """, llm_config = llm_config, code_execution_config = False,
                                             human_input_mode = "NEVER",
                                             is_termination_msg = lambda msg: ("terminate" in (msg.get("content") or "").lower()))

In [7]:
# Create Customer Proxy Agent

customer_proxy_agent = ConversableAgent(name="customer_proxy_agent", llm_config = False, 
                                        code_execution_config = False, human_input_mode = "ALWAYS",
                                        is_termination_msg = lambda msg: ("terminate" in (msg.get("content") or "").lower()))

In [8]:
# Create sequential chats

chats = [{"sender": onboarding_personal_information_agent, "recipient": customer_proxy_agent,
          "message": """Hello! Welcome to our platform. Could you tell me your name and location?""",
          "summary_method": "last_msg", "summary_args":{"summary_prompt": """
          Return the customer information as a JSON object only. Example: {"name": "Raji", "location": "Chennai"}"""},
          "max_turns": 2, "clear_history": True},
         
         {"sender": onboarding_topic_preference_agent, "recipient": customer_proxy_agent,
          "message": """Great! Could you tell me what topics, you are interested in reading about?""",
          "summary_method": "last_msg", "max_turns": 2, "clear_history": False},
         
          {"sender": customer_proxy_agent, "recipient": customer_engagement_agent, 
           "message": """Use the customer information and topic preference
           collected in the previous conversations.
           Generate personalized content based on those details.
           Do not use default or invented customer information.""",
           "max_turns": 2, "summary_method": "last_msg"}]

In [9]:
chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello! Welcome to our platform. Could you tell me your name and location?

--------------------------------------------------------------------------------


C:\Users\Raji\anaconda3\envs\ai_agents\Lib\site-packages\autogen\agentchat\chat.py:69: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  raji from chennai


customer_proxy_agent (to onboarding_personal_information_agent):

raji from chennai

--------------------------------------------------------------------------------
onboarding_personal_information_agent (to customer_proxy_agent):

CUSTOMER_DATA:{ "name": "raji", "location": "chennai" } TERMINATE

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (3bbaa5bd-00e5-4913-a2c0-d5ddea523493): Termination message condition on agent 'customer_proxy_agent' met

>>>>>>>> TERMINATING RUN (185d1c68-94c7-40e7-b1bb-3bf4430c5bb4): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics, you are interested in reading about?
Context: 
CUSTOMER_DATA:{ "name": "raji", "location": "chennai" } 

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...
customer_proxy_agent (to onboarding_topic_preference_agent):



--------------------------------------------------------------------------------
onboarding_topic_preference_agent (to customer_proxy_agent):

TOPIC_PREFERENCE: Please share your preferred reading topics (e.g., Technology, Sports, Thriller, Business, AI). TERMINATE

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Technology


customer_proxy_agent (to onboarding_topic_preference_agent):

Technology

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (6623e7e4-12cd-4609-92a9-5cc172f00e7f): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Use the customer information and topic preference
           collected in the previous conversations.
           Generate personalized content based on those details.
           Do not use default or invented customer information.
Context: 
CUSTOMER_DATA:{ "name": "raji", "location": "chennai" } 
Technology

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Raja, as a tech enthusiast in Chennai, you might be interested in 

In [10]:
# Extract summaries

summary = []

for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

CUSTOMER_DATA:{ "name": "raji", "location": "chennai" } 


Technology


Raja, as a tech enthusiast in Chennai, you might be interested in the latest advancements in artificial intelligence and 5G infrastructure. Recent trends include AI-driven healthcare solutions in local hospitals, leveraging machine learning for diagnostics, and the expansion of 5G networks across urban areas. These innovations are shaping the future of technology in India.  

For a thrilling take, here's a short story: *In the heart of Chennai, a hacker named Anu discovers a hidden data vault in a government database. With time ticking down, she must outwit AI security systems and prevent a catastrophic breach, blending tech prowess with suspense.*  






In [11]:
# Convert customer information into a DataFrame

import json
import re

# Get customer information from the first summary
data_str = summary[0]

# Extract the JSON object from the agent's response
match = re.search(r'\{.+\}', data_str, re.DOTALL)

if match:
    data_dict = json.loads(match.group())
else:
    raise ValueError(f"No JSON customer data found in: {data_str}")

# Create DataFrame
df = pd.DataFrame([data_dict])

# Add topic preference
df["TopicPreference"] = summary[1]

# Add customer engagement information
df["Engagement"] = summary[2]

df

,name,location,TopicPreference,Engagement
0,raji,chennai,Technology,"Raja, as a tech enthusiast in Chennai, you mig..."


In [12]:
# Display final customer information

print("Customer Onboarding Details")

display(df)

Customer Onboarding Details


,name,location,TopicPreference,Engagement
0,raji,chennai,Technology,"Raja, as a tech enthusiast in Chennai, you mig..."


In [13]:
# Display execution cost

for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 129, 'completion_tokens': 169, 'total_tokens': 298}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 129, 'completion_tokens': 169, 'total_tokens': 298}}}


{'usage_including_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 146, 'completion_tokens': 145, 'total_tokens': 291}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 146, 'completion_tokens': 145, 'total_tokens': 291}}}


{'usage_including_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 207, 'completion_tokens': 1049, 'total_tokens': 1256}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'qwen3:1.7b': {'cost': 0.0, 'prompt_tokens': 207, 'completion_tokens': 1049, 'total_tokens': 1256}}}


